# LUTM-1: Taichi CUDA simulator and enumerator

This backend evaluates independent blank-padded programs in parallel on the GPU. Programs use the convention `BBBBp#x`, where `B` is the LUTM blank symbol. Inputs are evaluated sequentially; programs within each batch run in parallel.

Taichi is initialized on **CUDA only**, with architecture fallback disabled. Run this notebook with **`LUTM-1` as the working directory** and a CUDA-capable GPU.

In [ ]:
from pathlib import Path
from time import perf_counter

required_files = ("lutm.py", "taichi_backend.py", "utils.py")
missing = [name for name in required_files if not (Path.cwd() / name).is_file()]
if missing:
    raise RuntimeError(
        "Run this notebook from the LUTM-1 repository root; missing: "
        + ", ".join(missing)
    )

from programs import get_program
from taichi_backend import (
    TaichiUTMSimulator,
    evaluate_program_taichi,
    find_exact_program_taichi,
    initialize_taichi_cuda,
)
from utils import (
    SimulatorConfig,
    TaskCases,
    padded_program_to_string,
    program_count,
)

initialize_taichi_cuda()
print("Taichi initialized on CUDA.")

## Run one explicit program

Edit the registered name, input, and budgets below. `program_width` is the fixed blank-padded program region and must be at least the literal program length. `batch_capacity` reserves GPU storage; it does not change the semantics of a single run.

In [ ]:
program_name = "bit_not"
input_bits = "10110"

program_width = 96
left_budget = 104
right_budget = 24
t_max = 20_000
batch_capacity = 512

program = get_program(program_name).program
config = SimulatorConfig(
    program_width=program_width,
    left_budget=left_budget,
    right_budget=right_budget,
    t_max=t_max,
)
simulator = TaichiUTMSimulator(config, batch_capacity=batch_capacity)
print(
    f"Ready: width={program_width}, tape={config.tape_size}, "
    f"T_max={t_max:,}, batch capacity={batch_capacity:,}"
)

In [ ]:
encoded = simulator.encode_programs([program])
padded = padded_program_to_string(
    encoded[0],
    blank_id=simulator.table.blank_id,
    zero_id=simulator.table.zero_id,
    one_id=simulator.table.one_id,
)

started = perf_counter()
result = simulator.simulate_one(program, input_bits)
elapsed = perf_counter() - started
output = result.output_strings()[0]
reason = result.invalid_reason_names()[0]
registered = get_program(program_name)
expected = registered.target(input_bits) if registered.oracle is not None else None

print(f"initial tape: {padded}#{input_bits}")
print(f"output:       {output!r}")
print(f"expected:     {expected!r}")
print(f"exact:        {expected is not None and not bool(result.invalid[0]) and output == expected}")
print(f"halted:       {bool(result.halted[0])}")
print(f"invalid:      {bool(result.invalid[0])} ({reason})")
print(f"T:            {int(result.T[0]):,}")
print(
    f"space L/R:    {int(result.left_space_used[0]):,} / "
    f"{int(result.right_space_used[0]):,}"
)
print(f"final head:   {int(result.final_heads[0]):,}")
print(f"final state:  {simulator.table.states[int(result.final_state_ids[0])]}")
print(f"elapsed:      {elapsed:.4f} s")

## Evaluate a program on input/target pairs

Edit the two lists directly. They must have the same length, inputs must be unique, and targets must be nonempty binary strings. Each input launches the selected program on the same CUDA simulator.

In [ ]:
task_program_name = "bit_not"
inputs = ["0", "1", "00", "01", "10", "11", "101"]
targets = ["1", "0", "11", "10", "01", "00", "010"]

task = TaskCases(inputs, targets)
task_program = get_program(task_program_name).program
print(f"Evaluating {task_program_name!r} on {len(task)} cases...", flush=True)

In [ ]:
started = perf_counter()
evaluations = evaluate_program_taichi(simulator, task_program, task)

print(f"{'input':>8}  {'target':>8}  {'output':>8}  {'exact':>5}  {'reason':>22}  {'T':>10}")
for case in evaluations:
    print(
        f"{case.input_bits!r:>8}  {case.target!r:>8}  {case.output!r:>8}  "
        f"{str(case.exact):>5}  {case.invalid_reason:>22}  {case.T:>10,}"
    )
print(f"\nall exact: {all(case.exact for case in evaluations)}")
print(f"elapsed:   {perf_counter() - started:.4f} s")

## Enumerate programs in growing length

The search order is empty, `0`, `1`, `00`, `01`, `10`, `11`, `000`, ... . On the fixed tape this appears as `BBBBB`, `BBBB0`, `BBBB1`, `BBB00`, ... . GPU threads independently simulate the programs in each ordinal batch.

With `stop_first_exact=True`, enumeration ends as soon as the first batch containing an exact program is evaluated. Ordinal enumeration supports widths up to 62; wider explicit programs can still be simulated above.

In [ ]:
search_inputs = ["0", "1", "01", "10"]
search_targets = ["0", "1", "01", "10"]

search_program_width = 5
search_left_budget = 7
search_right_budget = 8
search_t_max = 1_000
search_batch_capacity = 64
batch_size = 8
max_programs = None
stop_first_exact = True
print_every_batches = 1

search_config = SimulatorConfig(
    program_width=search_program_width,
    left_budget=search_left_budget,
    right_budget=search_right_budget,
    t_max=search_t_max,
)
search_simulator = TaichiUTMSimulator(
    search_config,
    batch_capacity=search_batch_capacity,
)
search_task = TaskCases(search_inputs, search_targets)

total = program_count(search_program_width)
limit = total if max_programs is None else min(total, max_programs)
print(
    f"Searching up to {limit:,} programs in batches of {batch_size:,}; "
    f"stop_first_exact={stop_first_exact}",
    flush=True,
)

In [ ]:
found = find_exact_program_taichi(
    search_simulator,
    search_task,
    batch_size=batch_size,
    max_programs=max_programs,
    stop_first_exact=stop_first_exact,
    print_every_batches=print_every_batches,
)

if found is None:
    print("No exact program exists within the selected search limit and budgets.")
else:
    print("\nFirst exact program")
    print(f"program:            {found.program!r}")
    print(f"padded:             {found.padded_program}")
    print(f"effective length:   {found.effective_length}")
    print(f"ordinal:            {found.program_ordinal:,}")
    print(f"evaluated programs: {found.evaluated_programs:,}")
    print(f"elapsed:            {found.elapsed_seconds:.3f} s")